# Financial Modeling Prep (FMP) API 全功能演示 

本 Notebook 覆盖 FMP Premium 的 **17 个主要类别**，每个类别都附带可运行的示例。

**前提条件**: `.env` 文件中已设置 `FMP_API_KEY`

In [1]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('FMP_API_KEY')
BASE = 'https://financialmodelingprep.com/stable'

def fmp(endpoint, **params):
    """通用 FMP API 请求函数，自动处理错误"""
    params['apikey'] = API_KEY
    url = f'{BASE}/{endpoint}'
    r = requests.get(url, params=params, timeout=30)
    if r.status_code == 402:
        print(f'⚠️ 402 Payment Required — 该 endpoint 需要更高级别订阅: {endpoint}')
        return pd.DataFrame()
    if r.status_code == 403:
        print(f'⚠️ 403 Forbidden — 无权访问: {endpoint}')
        return pd.DataFrame()
    if r.status_code != 200:
        print(f'⚠️ HTTP {r.status_code} for {endpoint}: {r.text[:200]}')
        return pd.DataFrame()
    data = r.json()
    if isinstance(data, list):
        return pd.DataFrame(data)
    elif isinstance(data, dict):
        return data
    return data

print(f'API Key loaded: {API_KEY[:8]}...')

API Key loaded: Y2OW5BnK...


---
## 1. 市场行情与实时数据 (Real-time & Quotes)
覆盖全球 70,000+ 股票的实时报价、盘后/盘前行情、批量报价和价格变化统计。

In [2]:
# 1a. 实时报价 (Real-time Quote)
quote = fmp('quote', symbol='AAPL')
quote[['symbol','name','price','change','changePercentage','dayLow','dayHigh','volume','marketCap']]

,symbol,name,price,change,changePercentage,dayLow,dayHigh,volume,marketCap
0,AAPL,Apple Inc.,270.23,6.83,2.59301,266.72,272.28,55211089,3.971821e+12


In [3]:
# 1b. 简短报价 (Quote Short)
fmp('quote-short', symbol='AAPL')

,symbol,price,change,volume
0,AAPL,270.23,6.83,55211089


In [4]:
# 1c. 盘后交易 (Aftermarket Trade)
fmp('aftermarket-trade', symbol='AAPL')

,symbol,price,tradeSize,timestamp
0,AAPL,270.6,None,1776470398000


In [5]:
# 1d. 盘后报价 (Aftermarket Quote)
fmp('aftermarket-quote', symbol='AAPL')

,symbol,bidSize,bidPrice,askSize,askPrice,volume,timestamp
0,AAPL,20,270.11,1,270.63,6.143616e+07,1776470399000


In [6]:
# 1e. 价格变化统计 (Price Change - 日/周/月/年维度)
fmp('stock-price-change', symbol='AAPL')

,symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
0,AAPL,2.59301,4.04266,8.11795,5.75275,7.11086,-0.599573,39.89957,62.32955,100.40789,905.69408,210441.48812


In [7]:
# 1f. 批量报价 (Batch Quotes)
batch = fmp('batch-quote', symbols='AAPL,MSFT,GOOGL,AMZN,NVDA')
batch[['symbol','name','price','change','volume']]

,symbol,name,price,change,volume
0,AAPL,Apple Inc.,270.23,6.83,55211089
1,MSFT,Microsoft Corporation,422.79,2.53,47754221
2,GOOGL,Alphabet Inc.,341.68,5.66,25487532
3,AMZN,"Amazon.com, Inc.",250.56,0.86,51829799
4,NVDA,NVIDIA Corporation,201.68,3.33,153662565


In [8]:
# 1g. 批量简短报价
fmp('batch-quote-short', symbols='AAPL,MSFT,GOOGL')

,symbol,price,change,volume
0,AAPL,270.23,6.83,55211089
1,MSFT,422.79,2.53,47754221
2,GOOGL,341.68,5.66,25487532


---
## 2. 财务报表 (Financial Statements - 深度 30+ 年)
利润表、资产负债表、现金流量表 — 年度/季度/TTM，以及原始报表和10-K JSON导出。

In [9]:
# 2a. 利润表 (Income Statement) - 年度
income = fmp('income-statement', symbol='AAPL', period='annual', limit=5)
income[['date','fiscalYear','revenue','grossProfit','netIncome','eps','epsDiluted']]

,date,fiscalYear,revenue,grossProfit,netIncome,eps,epsDiluted
0,2025-09-27,2025,416161000000,195201000000,112010000000,7.49,7.46
1,2024-09-28,2024,391035000000,180683000000,93736000000,6.11,6.08
2,2023-09-30,2023,383285000000,169148000000,96995000000,6.16,6.13
3,2022-09-24,2022,394328000000,170782000000,99803000000,6.15,6.11
4,2021-09-25,2021,365817000000,152836000000,94680000000,5.67,5.61


In [10]:
# 2b. 资产负债表 (Balance Sheet) - 季度
bs = fmp('balance-sheet-statement', symbol='AAPL', period='quarter', limit=4)
bs.head()

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,cashAndCashEquivalents,shortTermInvestments,...,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt
0,2025-12-27,AAPL,USD,0000320193,2026-01-30,2026-01-30 06:01:32,2026,Q1,45317000000,21590000000,...,0,-4854000000,0,88190000000,88190000000,0,379297000000,99478000000,90509000000,45192000000
1,2025-09-27,AAPL,USD,0000320193,2025-10-31,2025-10-31 06:01:26,2025,Q4,35934000000,18763000000,...,0,-5571000000,0,73733000000,73733000000,0,359241000000,96486000000,112377000000,76443000000
2,2025-06-28,AAPL,USD,0000320193,2025-08-01,2025-08-01 06:00:42,2025,Q3,36269000000,19103000000,...,0,-6369000000,0,65830000000,65830000000,0,331495000000,96717000000,101698000000,65429000000
3,2025-03-29,AAPL,USD,0000320193,2025-05-02,2025-05-02 06:00:46,2025,Q2,28162000000,20336000000,...,0,-6363000000,0,66796000000,66796000000,0,331233000000,104760000000,98186000000,70024000000


In [11]:
# 2c. 现金流量表 (Cash Flow Statement) - 年度
cf = fmp('cash-flow-statement', symbol='BABA', period='annual', limit=5)
cf.head()

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,netIncome,depreciationAndAmortization,...,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid
0,2025-03-31,BABA,CNY,0001577552,2025-06-26,2025-06-26 06:31:51,2025,FY,130508124000,31098371000,...,-57122379000,-1462462000,-100370222000,181762588000,282132810000,164820058000,-86661346000,78158712000,0,0
1,2024-03-31,BABA,CNY,0001577552,2024-05-23,2024-05-23 09:14:49,2024,FY,80358477000,45283964000,...,-104805328000,-12336028000,40595701000,279089896000,238494195000,184006915000,-33183987000,150822928000,0,0
2,2023-03-31,BABA,CNY,0001577552,2023-07-21,2023-07-21 07:43:34,2023,FY,72562841000,38186334000,...,-64073542000,-15934845000,-17308865000,226019902000,243328767000,199900323000,-34377508000,165522815000,26476000000,5637000000
3,2022-03-31,BABA,CNY,0001577552,2022-07-26,2022-07-26 07:49:18,2022,FY,60929891000,44967497000,...,-68132243000,2620557000,-115663617000,225367358000,341030974000,140387842000,-52438314000,87949528000,21474000000,4886000000
4,2021-03-31,BABA,CNY,0001577552,2021-07-27,2021-07-27 07:55:28,2021,FY,145017836000,45413348000,...,29214281000,21922939000,38974894000,354939789000,315964895000,223628178000,-41665083000,181963095000,20898000000,4101000000


### CYATY (CyberArk) DCF 分析示例
利用 FMP 多个 endpoint 对 CYATY 进行完整的 DCF 估值分析。

In [12]:
# CYATY - 公司概况 (Company Profile)
cyaty_profile = fmp('profile', symbol='CYATY')
if isinstance(cyaty_profile, pd.DataFrame) and len(cyaty_profile) > 0:
    cols = ['symbol','companyName','currency','exchangeShortName','marketCap','price','sector','industry','country']
    available = [c for c in cols if c in cyaty_profile.columns]
    display(cyaty_profile[available])
else:
    print(cyaty_profile)

,symbol,companyName,currency,marketCap,price,sector,industry,country
0,CYATY,"Contemporary Amperex Technology Co., Limited",USD,1268187192747,22.46,Industrials,Electrical Equipment & Parts,CN


In [13]:
# CYATY - 现金流量表 (最近10年)
cyaty_cf = fmp('cash-flow-statement', symbol='CYATY', period='annual', limit=10)
if isinstance(cyaty_cf, pd.DataFrame) and len(cyaty_cf) > 0:
    fcf_cols = ['date','fiscalYear','operatingCashFlow','capitalExpenditure','freeCashFlow',
                'netIncome','dividendsPaid','stockBasedCompensation']
    available = [c for c in fcf_cols if c in cyaty_cf.columns]
    display(cyaty_cf[available])
else:
    print('未获取到 CYATY 现金流数据')

,date,fiscalYear,operatingCashFlow,capitalExpenditure,freeCashFlow,netIncome,stockBasedCompensation
0,2025-12-31,2025,131808765000,-41190888000,90617877000,70234170000,0
1,2024-12-31,2024,96990345000,-31179943000,65810402000,54006794000,0
2,2023-12-31,2023,92826124000,-33624897000,59201227000,46761034000,0
3,2022-12-31,2022,61208843000,-48215268000,12993575000,33457144000,0
4,2021-12-31,2021,42908009000,-43767771000,-859762000,17860730000,0


In [14]:
# CYATY - FMP 内置 DCF 估值
cyaty_dcf = fmp('discounted-cash-flow', symbol='CYATY')
print('=== FMP 实时 DCF ===')
print(cyaty_dcf)

print()

# 杠杆 DCF
cyaty_ldcf = fmp('levered-discounted-cash-flow', symbol='CYATY')
print('=== FMP 杠杆 DCF ===')
print(cyaty_ldcf)

=== FMP 实时 DCF ===
  symbol        date        dcf  Stock Price
0  CYATY  2026-04-18  20.709955        22.46

=== FMP 杠杆 DCF ===
  symbol        date        dcf  Stock Price
0  CYATY  2026-04-18  52.023369        22.46


In [ ]:
# CYATY - 核心估值指标 (Key Metrics)
cyaty_km = fmp('key-metrics', symbol='CYATY', period='annual', limit=10)
if isinstance(cyaty_km, pd.DataFrame) and len(cyaty_km) > 0:
    val_cols = ['date','fiscalYear','marketCap','enterpriseValue','peRatio','priceToFreeCashFlowsRatio',
                'freeCashFlowPerShare','freeCashFlowYield','roic','roe']
    available = [c for c in val_cols if c in cyaty_km.columns]
    display(cyaty_km[available])
else:
    print('未获取到 CYATY key metrics')

In [ ]:
# CYATY - 自定义简易 DCF 计算 (基于 FCF 历史 + 14x/24x/34x 倍数)
import numpy as np

if isinstance(cyaty_cf, pd.DataFrame) and 'freeCashFlow' in cyaty_cf.columns:
    df = cyaty_cf[['date','fiscalYear','freeCashFlow']].dropna().sort_values('fiscalYear')
    print('=== CYATY FCF 历史 ===')
    for _, row in df.iterrows():
        print(f"  {row['fiscalYear']}:  FCF = {row['freeCashFlow']:>15,.0f}")

    # 取最近3年滚动平均
    recent_fcf = df.tail(3)['freeCashFlow'].values
    avg_fcf_3y = np.mean(recent_fcf)
    print(f'\n最近3年平均 FCF: {avg_fcf_3y:,.0f}')

    # 获取流通股数
    shares = None
    if isinstance(cyaty_km, pd.DataFrame) and 'marketCap' in cyaty_km.columns:
        # 从 profile 获取价格来推算
        if isinstance(cyaty_profile, pd.DataFrame) and 'price' in cyaty_profile.columns:
            price = cyaty_profile['price'].iloc[0]
            mcap = cyaty_km['marketCap'].iloc[0]
            if price and price > 0:
                shares = mcap / price

    if shares and shares > 0:
        avg_fcf_per_share = avg_fcf_3y / shares
        print(f'估算流通股数: {shares:,.0f}')
        print(f'3年平均 FCF/Share: {avg_fcf_per_share:.2f}')
        print(f'\n=== 倍数估值 (基于3年平均 FCF/Share) ===')
        for mult in [14, 24, 34]:
            val = avg_fcf_per_share * mult
            print(f'  {mult}x FCF → 每股价值 = {val:.2f}')
    else:
        print(f'\n=== 倍数估值 (基于3年平均总 FCF) ===')
        for mult in [14, 24, 34]:
            val = avg_fcf_3y * mult
            print(f'  {mult}x FCF → 企业价值 = {val:,.0f}')
else:
    print('无 FCF 数据，无法计算')

In [ ]:
# CYATY - FCF 增长趋势可视化
import plotly.graph_objects as go

if isinstance(cyaty_cf, pd.DataFrame) and 'freeCashFlow' in cyaty_cf.columns:
    df = cyaty_cf[['fiscalYear','freeCashFlow','operatingCashFlow','capitalExpenditure']].dropna().sort_values('fiscalYear')

    fig = go.Figure()
    fig.add_trace(go.Bar(x=df['fiscalYear'], y=df['operatingCashFlow'], name='经营现金流', marker_color='steelblue'))
    fig.add_trace(go.Bar(x=df['fiscalYear'], y=df['capitalExpenditure'], name='资本支出', marker_color='salmon'))
    fig.add_trace(go.Scatter(x=df['fiscalYear'], y=df['freeCashFlow'], mode='lines+markers',
                             name='自由现金流 (FCF)', line=dict(color='green', width=3)))
    fig.update_layout(title='CYATY - FCF 历史趋势', xaxis_title='财年', yaxis_title='金额',
                      barmode='group', template='plotly_white', height=450)
    fig.show()
else:
    print('无数据')

In [ ]:
# CYATY - 分析师目标价 vs DCF 估值对比
cyaty_pt = fmp('price-target-consensus', symbol='CYATY')
print('=== 分析师目标价共识 ===')
print(cyaty_pt)

print()

cyaty_rating = fmp('ratings-snapshot', symbol='CYATY')
print('=== FMP 综合评分 ===')
print(cyaty_rating)

In [20]:
# 2d. TTM (滚动十二个月)
income_ttm = fmp('income-statement-ttm', symbol='AAPL')
income_ttm.head()

⚠️ 402 Payment Required — 该 endpoint 需要更高级别订阅: income-statement-ttm


""


In [21]:
# 2e. 原始报表 (As Reported) - 按SEC原始格式
as_reported = fmp('income-statement-as-reported', symbol='AAPL', period='annual', limit=2)
as_reported.head()

,symbol,fiscalYear,period,reportedCurrency,date,data
0,AAPL,2025,FY,None,2025-09-26,{'revenuefromcontractwithcustomerexcludingasse...
1,AAPL,2024,FY,USD,2024-09-27,{'revenuefromcontractwithcustomerexcludingasse...


In [27]:
# 2f. 10-K 结构化 JSON — 深入探索年报内容
report_json = fmp('financial-reports-json', symbol='AAPL', year=2024, period='FY')

if isinstance(report_json, dict):
    print('='*60)
    print('顶层 keys:')
    for k in report_json.keys():
        v = report_json[k]
        if isinstance(v, dict):
            print(f'  📂 {k}  ({len(v)} sub-keys)')
        elif isinstance(v, list):
            print(f'  📋 {k}  (list, len={len(v)})')
        else:
            print(f'  🔹 {k} = {str(v)[:80]}')

    # 逐个子节展开一层，展示内部字段
    print('\n' + '='*60)
    print('各子节详细内容预览:\n')
    for k, v in report_json.items():
        if isinstance(v, dict):
            print(f'── {k} ──')
            for sub_k, sub_v in list(v.items())[:8]:
                val_repr = str(sub_v)[:100] if not isinstance(sub_v, dict) else f'dict({len(sub_v)} keys)'
                print(f'   {sub_k}: {val_repr}')
            if len(v) > 8:
                print(f'   ... 共 {len(v)} 个字段')
            print()
else:
    print('返回类型:', type(report_json))
    if isinstance(report_json, pd.DataFrame):
        print(f'行数: {len(report_json)}')
        report_json.head()

顶层 keys:
  🔹 symbol = AAPL
  🔹 period = FY
  🔹 year = 2024
  📋 Cover Page  (list, len=80)
  📋 Auditor Information  (list, len=6)
  📋 CONSOLIDATED STATEMENTS OF OPER  (list, len=26)
  📋 CONSOLIDATED STATEMENTS OF COMP  (list, len=16)
  📋 CONSOLIDATED BALANCE SHEETS  (list, len=36)
  📋 CONSOLIDATED BALANCE SHEETS (Pa  (list, len=6)
  📋 CONSOLIDATED STATEMENTS OF SHAR  (list, len=33)
  📋 CONSOLIDATED STATEMENTS OF CASH  (list, len=38)
  📋 Summary of Significant Accounti  (list, len=4)
  📋 Revenue  (list, len=4)
  📋 Earnings Per Share  (list, len=4)
  📋 Financial Instruments  (list, len=4)
  📋 Property, Plant and Equipment  (list, len=4)
  📋 Consolidated Financial Statemen  (list, len=4)
  📋 Income Taxes  (list, len=4)
  📋 Leases  (list, len=5)
  📋 Debt  (list, len=4)
  📋 Shareholders' Equity  (list, len=4)
  📋 Share-Based Compensation  (list, len=4)
  📋 Commitments, Contingencies and   (list, len=4)
  📋 Segment Information and Geograp  (list, len=4)
  📋 Pay vs Performance Disclosure  (lis

---
## 3. 估值与高级财务指标 (Valuation & Key Metrics)
P/E, P/S, ROE, ROIC 等 50+ 种指标、财务比率、企业价值和所有者收益。

In [23]:
# 3a. 核心指标 (Key Metrics) - 年度
km = fmp('key-metrics', symbol='AAPL', period='annual', limit=5)
km.head()

,symbol,date,fiscalYear,period,reportedCurrency,marketCap,enterpriseValue,evToSales,evToOperatingCashFlow,evToFreeCashFlow,...,averageInventory,daysOfSalesOutstanding,daysOfPayablesOutstanding,daysOfInventoryOutstanding,operatingCycle,cashConversionCycle,freeCashFlowToEquity,freeCashFlowToFirm,tangibleAssetValue,netCurrentAssetValue
0,AAPL,2025-09-27,2025,FY,USD,3818743810000,3895186810000,9.359807,34.940051,39.438140,...,6502000000,63.987988,115.400525,9.445465,73.433453,-41.967072,22324000000,1.052620e+11,73733000000,-137551000000
1,AAPL,2024-09-28,2024,FY,USD,3495160329570,3584276329570,9.166127,30.309980,32.941597,...,6808500000,61.832560,119.658477,12.642571,74.475130,-45.183347,19691000000,1.173970e+11,56950000000,-155043000000
2,AAPL,2023-09-30,2023,FY,USD,2695569789510,2789534789510,7.277965,25.234839,28.011877,...,5638500000,58.075649,106.721468,10.791292,68.866941,-37.854527,5619000000,8.407409e+10,62146000000,-146871000000
3,AAPL,2022-09-24,2022,FY,USD,2439367314090,2548201314090,6.462136,20.861076,22.865513,...,5763000000,56.400205,104.685277,8.075698,64.475903,-40.209374,2609000000,1.305870e+11,50672000000,-166678000000
4,AAPL,2021-09-25,2021,FY,USD,2453750882240,2555332882240,6.985276,24.561534,27.490591,...,5320500000,51.390969,93.851071,11.276593,62.667561,-31.183510,-8629000000,1.261382e+11,63090000000,-153076000000


In [24]:
# 3b. 核心指标 TTM
km_ttm = fmp('key-metrics-ttm', symbol='AAPL')
km_ttm.head()

,symbol,marketCap,enterpriseValueTTM,evToSalesTTM,evToOperatingCashFlowTTM,evToFreeCashFlowTTM,evToEBITDATTM,netDebtToEBITDATTM,currentRatioTTM,incomeQualityTTM,...,averageInventoryTTM,daysOfSalesOutstandingTTM,daysOfPayablesOutstandingTTM,daysOfInventoryOutstandingTTM,operatingCycleTTM,cashConversionCycleTTM,freeCashFlowToEquityTTM,freeCashFlowToFirmTTM,tangibleAssetValueTTM,netCurrentAssetValueTTM
0,AAPL,3.971821e+12,4.017013e+12,9.221432,29.651978,32.572838,26.258589,0.295413,0.973745,1.150242,...,5796500000,58.920566,112.282119,9.345311,68.265876,-44.016243,78132000000,104050000000,88190000000,-133003000000


In [25]:
# 3c. 财务比率 (Financial Ratios)
ratios = fmp('ratios', symbol='AAPL', period='annual', limit=5)
ratios.head()

,symbol,date,fiscalYear,period,reportedCurrency,grossProfitMargin,ebitMargin,ebitdaMargin,operatingProfitMargin,pretaxProfitMargin,...,operatingCashFlowPerShare,capexPerShare,freeCashFlowPerShare,netIncomePerEBT,ebtPerEbit,priceToFairValue,debtToMarketCap,effectiveTaxRate,enterpriseValueMultiple,dividendPerShare
0,AAPL,2025-09-27,2025,FY,USD,0.469052,0.318937,0.347046,0.319708,0.318937,...,7.457738,0.850587,6.607151,0.843900,0.997587,51.791515,0.025835,0.156100,26.969935,1.031609
1,AAPL,2024-09-28,2024,FY,USD,0.462063,0.315790,0.345059,0.315102,0.315790,...,7.706965,0.615689,7.091276,0.759088,1.002183,61.372438,0.030508,0.240912,26.563969,0.992845
2,AAPL,2023-09-30,2023,FY,USD,0.441311,0.307001,0.337055,0.298214,0.296740,...,7.021175,0.696064,6.325110,0.852808,0.995057,43.374791,0.041211,0.147192,21.592832,0.954318
3,AAPL,2022-09-24,2022,FY,USD,0.433096,0.309473,0.337633,0.302887,0.302040,...,7.532763,0.660337,6.872426,0.837955,0.997204,48.140340,0.049221,0.162045,19.139549,0.915209
4,AAPL,2021-09-25,2021,FY,USD,0.417794,0.305759,0.336605,0.297824,0.298529,...,6.229346,0.663722,5.565624,0.866977,1.002368,38.892865,0.050828,0.133023,20.752119,0.866221


In [26]:
# 3d. 财务比率 TTM
ratios_ttm = fmp('ratios-ttm', symbol='AAPL')
ratios_ttm.head()

,symbol,grossProfitMarginTTM,ebitMarginTTM,ebitdaMarginTTM,operatingProfitMarginTTM,pretaxProfitMarginTTM,continuousOperationsProfitMarginTTM,netProfitMarginTTM,bottomLineProfitMarginTTM,receivablesTurnoverTTM,...,operatingCashFlowPerShareTTM,capexPerShareTTM,freeCashFlowPerShareTTM,netIncomePerEBTTTM,ebtPerEbitTTM,priceToFairValueTTM,debtToMarketCapTTM,effectiveTaxRateTTM,enterpriseValueMultipleTTM,dividendPerShareTTM
0,AAPL,0.473253,0.324016,0.351178,0.32384,0.324016,0.270368,0.270368,0.270368,6.194781,...,9.185689,0.823696,8.361993,0.834428,1.000546,45.191005,0.022788,0.165572,26.258589,1.04


In [28]:
# 3e. 企业价值 (Enterprise Value)
ev = fmp('enterprise-values', symbol='AAPL', period='annual', limit=5)
ev.head()

,symbol,date,stockPrice,numberOfShares,marketCapitalization,minusCashAndCashEquivalents,addTotalDebt,enterpriseValue
0,AAPL,2025-09-27,255.46,14948500000,3818743810000,35934000000,112377000000,3895186810000
1,AAPL,2024-09-28,227.79,15343783000,3495160329570,29943000000,119059000000,3584276329570
2,AAPL,2023-09-30,171.21,15744231000,2695569789510,29965000000,123930000000,2789534789510
3,AAPL,2022-09-24,150.43,16215963000,2439367314090,23646000000,132480000000,2548201314090
4,AAPL,2021-09-25,146.92,16701272000,2453750882240,34940000000,136522000000,2555332882240


In [ ]:
# 3f. 所有者收益 (Owner Earnings)
oe = fmp('owner-earnings', symbol='AAPL', limit=5)
oe.head()

In [ ]:
# 3g. 财务评分 (Financial Scores - Altman Z-Score & Piotroski)
scores = fmp('financial-scores', symbol='AAPL')
scores

---
## 4. 现金流折现模型 (DCF Valuation)
实时 DCF、自定义 DCF 和杠杆 DCF。

In [ ]:
# 4a. 实时 DCF 估值
dcf = fmp('discounted-cash-flow', symbol='AAPL')
dcf

In [ ]:
# 4b. 杠杆 DCF (Levered DCF)
ldcf = fmp('levered-discounted-cash-flow', symbol='AAPL')
ldcf

In [ ]:
# 4c. 自定义 DCF (Custom DCF Advanced)
# 可以传入自定义参数，这里用默认值
custom_dcf = fmp('custom-discounted-cash-flow', symbol='AAPL')
custom_dcf

In [ ]:
# 4d. 自定义杠杆 DCF
custom_ldcf = fmp('custom-levered-discounted-cash-flow', symbol='AAPL')
custom_ldcf

---
## 5. 收入构成拆解 (Revenue Segmentation)
产品线拆分和地理区域拆分。

In [ ]:
# 5a. 产品细分 (Product Segmentation)
prod_seg = fmp('revenue-product-segmentation', symbol='AAPL')
prod_seg

In [ ]:
# 5b. 地理位置细分 (Geographic Segmentation)
geo_seg = fmp('revenue-geographic-segmentation', symbol='AAPL')
geo_seg

---
## 6. 分析师预期与评级 (Analyst Estimates & Ratings)
盈利预期、目标价和评级建议。

In [29]:
# 6a. 盈利预期 (Financial Estimates)
est = fmp('analyst-estimates', symbol='AAPL', period='annual', page=0, limit=5)
est.head()

,symbol,date,revenueLow,revenueHigh,revenueAvg,ebitdaLow,ebitdaHigh,ebitdaAvg,ebitLow,ebitHigh,...,netIncomeHigh,netIncomeAvg,sgaExpenseLow,sgaExpenseHigh,sgaExpenseAvg,epsAvg,epsHigh,epsLow,numAnalystsRevenue,numAnalystsEps
0,AAPL,2030-09-27,602061271079,659243056109,627849333333,217304958132,237943863172,226612771236,201646448083,220798159712,...,208912346955,196161355444,38750143588,42430503856,40409926676,13.07333,13.92313,12.37527,16,7
1,AAPL,2029-09-27,463249649440,507247566500,483092000000,167202991642,183083377882,174364789560,155154717423,169890798490,...,184524162591,173261637022,29815886348,32647700460,31092988813,11.54716,12.29776,10.93059,10,15
2,AAPL,2028-09-27,524242643201,528680401180,526461522191,189217495136,190819237115,190018366126,175582904952,177069228981,...,168380308993,144807664660,33741545379,34027170393,33884357886,10.21077,11.22184,8.96953,19,17
3,AAPL,2027-09-27,481838357146,530134299504,495841678050,173912306032,191343999842,178966594041,161380573599,177556178478,...,147988775722,133834797140,31012301277,34120746864,31913589438,9.26508,9.86283,8.39835,31,31
4,AAPL,2026-09-27,439857987044,481138994586,464391217404,158760123014,173659881638,167615023425,147320223056,161146338343,...,133693950936,124662378470,28310341447,30967288583,29889360469,8.47624,8.91014,7.83043,32,31


In [30]:
# 6b. 目标价共识 (Price Target Consensus)
ptc = fmp('price-target-consensus', symbol='AAPL')
ptc

,symbol,targetHigh,targetLow,targetConsensus,targetMedian
0,AAPL,350,239,315.91,325


In [31]:
# 6c. 目标价汇总 (Price Target Summary)
pts = fmp('price-target-summary', symbol='AAPL')
pts

,symbol,lastMonthCount,lastMonthAvgPriceTarget,lastQuarterCount,lastQuarterAvgPriceTarget,lastYearCount,lastYearAvgPriceTarget,allTimeCount,allTimeAvgPriceTarget,publishers
0,AAPL,1,300,12,309.29,50,286.86,233,221.47,"[""TheFly"",""StreetInsider"",""Benzinga"",""Pulse 2...."


In [ ]:
# 6d. 评级建议 (Stock Grades)
grades = fmp('grades', symbol='AAPL', limit=10)
grades.head()

In [ ]:
# 6e. 评级汇总 (Grades Summary/Consensus)
gc = fmp('grades-consensus', symbol='AAPL')
gc

In [ ]:
# 6f. FMP综合评分 (Ratings Snapshot)
rating = fmp('ratings-snapshot', symbol='AAPL')
rating

---
## 7. 财报电话会议文本 (Earnings Transcripts)
全文本记录，覆盖完整历史。

In [ ]:
# 7a. 查询可用的 Transcript 日期
tr_dates = fmp('earning-call-transcript-dates', symbol='AAPL')
tr_dates.head(10)

In [ ]:
# 7b. 获取具体某季度的 Earning Call Transcript (截取前2000字)
transcript = fmp('earning-call-transcript', symbol='AAPL', year=2024, quarter=4)
if isinstance(transcript, pd.DataFrame) and len(transcript) > 0:
    text = transcript.iloc[0].get('content', '')
    print(f'Transcript length: {len(text)} chars')
    print(text[:2000])
else:
    print(transcript)

In [ ]:
# 7c. 最新 Transcripts
latest_tr = fmp('earning-call-transcript-latest', page=0, limit=5)
latest_tr.head()

---
## 8. 机构持仓 (Form 13F - Institutional Ownership)
追踪贝莱德、先锋领航、巴菲特等大玩家的持仓变动。

In [ ]:
# 8a. 最新机构持仓报告
inst_latest = fmp('institutional-ownership/latest', page=0, limit=10)
inst_latest.head()

In [ ]:
# 8b. 巴菲特 (Berkshire) 13F 提交日期
bh_dates = fmp('institutional-ownership/dates', cik='0001067983')
bh_dates.head()

In [ ]:
# 8c. 提取巴菲特最新持仓明细
bh_holdings = fmp('institutional-ownership/extract', cik='0001067983', year=2024, quarter=4)
if isinstance(bh_holdings, pd.DataFrame) and len(bh_holdings) > 0:
    bh_holdings.head(15)
else:
    print(bh_holdings)

In [ ]:
# 8d. AAPL 的机构持仓汇总 (Positions Summary)
pos_sum = fmp('institutional-ownership/symbol-positions-summary', symbol='AAPL', year=2024, quarter=4)
pos_sum

In [ ]:
# 8e. 巴菲特持仓行业分布
bh_industry = fmp('institutional-ownership/holder-industry-breakdown', cik='0001067983', year=2024, quarter=4)
bh_industry.head(10)

---
## 9. 内幕交易 (Insider Trading)
实时追踪 CEO、CFO 及大股东的个人交易。

In [ ]:
# 9a. 最新内幕交易
insider_latest = fmp('insider-trading/latest', page=0, limit=10)
insider_latest.head()

In [ ]:
# 9b. 搜索特定公司内幕交易 (AAPL)
insider_aapl = fmp('insider-trading/search', symbol='AAPL', limit=10)
insider_aapl.head()

In [ ]:
# 9c. 按人名搜索 (Zuckerberg)
insider_zuck = fmp('insider-trading/reporting-name', name='Zuckerberg')
insider_zuck.head()

In [ ]:
# 9d. 内幕交易统计 (净买入/卖出)
insider_stats = fmp('insider-trading/statistics', symbol='AAPL')
insider_stats

---
## 10. 历史股价与技术分析 (Charts & Technicals)
分时数据、除权除息调整价格、SMA/EMA/RSI/ADX 等技术指标。

In [ ]:
# 10a. 日线 (EOD Full) - 最近30天
eod = fmp('historical-price-eod/full', symbol='AAPL', **{'from': '2025-03-01', 'to': '2025-04-17'})
eod.head(10)

In [ ]:
# 10b. 日线轻量版 (Light)
eod_light = fmp('historical-price-eod/light', symbol='AAPL', **{'from': '2025-03-01', 'to': '2025-04-17'})
eod_light.head(10)

In [ ]:
# 10c. 除权除息调整价格 (Dividend Adjusted)
div_adj = fmp('historical-price-eod/dividend-adjusted', symbol='AAPL', **{'from': '2024-01-01', 'to': '2024-12-31'})
div_adj.head(5)

In [ ]:
# 10d. 分时数据 - 5分钟级别
intraday_5m = fmp('historical-chart/5min', symbol='AAPL', **{'from': '2025-04-16', 'to': '2025-04-17'})
intraday_5m.head(10)

In [ ]:
# 10e. 分时数据 - 1小时级别
intraday_1h = fmp('historical-chart/1hour', symbol='AAPL', **{'from': '2025-04-14', 'to': '2025-04-17'})
intraday_1h.head(10)

In [ ]:
# 10f. 技术指标 - SMA (Simple Moving Average)
sma = fmp('technical-indicators/sma', symbol='AAPL', periodLength=20, timeframe='1day')
sma.head(10)

In [ ]:
# 10g. 技术指标 - EMA (Exponential Moving Average)
ema = fmp('technical-indicators/ema', symbol='AAPL', periodLength=10, timeframe='1day')
ema.head(10)

In [ ]:
# 10h. 技术指标 - RSI (Relative Strength Index)
rsi = fmp('technical-indicators/rsi', symbol='AAPL', periodLength=14, timeframe='1day')
rsi.head(10)

In [ ]:
# 10i. 技术指标 - ADX (Average Directional Index)
adx = fmp('technical-indicators/adx', symbol='AAPL', periodLength=14, timeframe='1day')
adx.head(10)

---
## 11. 另类数据 (Alternative Data)
政界交易、ESG 评分、众筹数据和 COT 报告。

In [ ]:
# 11a. 参议院交易 (Senate Trading)
senate = fmp('senate-latest', page=0, limit=10)
senate.head()

In [ ]:
# 11b. 按股票查看参议院交易
senate_aapl = fmp('senate-trades', symbol='AAPL')
senate_aapl.head()

In [ ]:
# 11c. 众议院交易 (House Trading)
house = fmp('house-latest', page=0, limit=10)
house.head()

In [ ]:
# 11d. ESG 评分
esg = fmp('esg-disclosures', symbol='AAPL')
esg.head()

In [ ]:
# 11e. ESG 评级
esg_rating = fmp('esg-ratings', symbol='AAPL')
esg_rating

In [ ]:
# 11f. 众筹数据 (Crowdfunding)
crowd = fmp('crowdfunding-offerings-latest', page=0, limit=5)
crowd.head()

In [ ]:
# 11g. COT 报告 (Commitment of Traders)
cot_list = fmp('commitment-of-traders-list')
cot_list.head(10)

In [ ]:
# 11h. COT 分析
cot = fmp('commitment-of-traders-analysis', **{'from': '2025-01-01', 'to': '2025-04-01'})
if isinstance(cot, pd.DataFrame):
    cot.head(10)
else:
    print(cot)

---
## 12. 宏观经济数据 (Economics)
国债收益率、GDP、CPI、失业率、美联储利率等。

In [ ]:
# 12a. 国债收益率 (Treasury Rates)
treasury = fmp('treasury-rates', **{'from': '2025-03-01', 'to': '2025-04-17'})
treasury.head(10)

In [ ]:
# 12b. GDP 数据
gdp = fmp('economic-indicators', name='GDP', **{'from': '2020-01-01', 'to': '2025-12-31'})
gdp.head(10)

In [ ]:
# 12c. CPI (消费者物价指数)
cpi = fmp('economic-indicators', name='CPI', **{'from': '2023-01-01', 'to': '2025-12-31'})
cpi.head(10)

In [ ]:
# 12d. 失业率 (Unemployment Rate)
unemp = fmp('economic-indicators', name='unemploymentRate', **{'from': '2023-01-01', 'to': '2025-12-31'})
unemp.head(10)

In [ ]:
# 12e. 经济数据发布日历
econ_cal = fmp('economic-calendar', **{'from': '2025-04-01', 'to': '2025-04-30'})
econ_cal.head(10)

In [ ]:
# 12f. 市场风险溢价 (Market Risk Premium)
mrp = fmp('market-risk-premium')
mrp.head(10)

---
## 13. 市场日历 (Market Calendars)
财报日历、分红日历、拆股日历和 IPO 日历。

In [ ]:
# 13a. 财报日历 (Earnings Calendar)
earn_cal = fmp('earnings-calendar', **{'from': '2025-04-14', 'to': '2025-04-25'})
earn_cal.head(10)

In [ ]:
# 13b. 个股财报历史 (Earnings per Company)
earn_aapl = fmp('earnings', symbol='AAPL', limit=8)
earn_aapl.head()

In [ ]:
# 13c. 分红日历 (Dividends Calendar)
div_cal = fmp('dividends-calendar', **{'from': '2025-04-01', 'to': '2025-04-30'})
div_cal.head(10)

In [ ]:
# 13d. 个股分红历史
div_aapl = fmp('dividends', symbol='AAPL')
div_aapl.head(10)

In [ ]:
# 13e. 拆股日历 (Stock Splits Calendar)
split_cal = fmp('splits-calendar', **{'from': '2024-01-01', 'to': '2025-04-17'})
split_cal.head(10)

In [ ]:
# 13f. 个股拆股历史
split_aapl = fmp('splits', symbol='AAPL')
split_aapl

In [ ]:
# 13g. IPO 日历
ipo_cal = fmp('ipos-calendar', **{'from': '2025-03-01', 'to': '2025-04-30'})
ipo_cal.head(10)

---
## 14. 基金与 ETF 数据 (ETF & Mutual Funds)
持仓构成、行业/国家权重和资产敞口。

In [ ]:
# 14a. ETF 持仓 (SPY Top Holdings)
spy_hold = fmp('etf/holdings', symbol='SPY')
spy_hold.head(15)

In [ ]:
# 14b. ETF 基本信息
spy_info = fmp('etf/info', symbol='SPY')
spy_info

In [ ]:
# 14c. ETF 行业权重 (Sector Weighting)
spy_sector = fmp('etf/sector-weightings', symbol='SPY')
spy_sector

In [ ]:
# 14d. ETF 国家分布 (Country Allocation)
vwo_country = fmp('etf/country-weightings', symbol='VWO')
vwo_country.head(10)

In [ ]:
# 14e. 资产敞口 - 哪些 ETF 持有 AAPL?
aapl_etfs = fmp('etf/asset-exposure', symbol='AAPL')
aapl_etfs.head(15)

---
## 15. 指数数据 (Indexes)
S&P 500, Nasdaq 100, Dow Jones 成分股及历史走势。

In [ ]:
# 15a. S&P 500 成分股
sp500 = fmp('sp500-constituent')
print(f'S&P 500 成分股数量: {len(sp500)}')
sp500.head(10)

In [ ]:
# 15b. Nasdaq 100 成分股
nasdaq = fmp('nasdaq-constituent')
print(f'Nasdaq 100 成分股数量: {len(nasdaq)}')
nasdaq.head(10)

In [ ]:
# 15c. Dow Jones 成分股
dow = fmp('dowjones-constituent')
dow.head(10)

In [ ]:
# 15d. S&P 500 指数历史走势
sp_hist = fmp('historical-price-eod/full', symbol='^GSPC', **{'from': '2025-01-01', 'to': '2025-04-17'})
sp_hist.head(10)

In [ ]:
# 15e. 指数实时报价
idx_quote = fmp('quote', symbol='^GSPC')
idx_quote.head()

In [ ]:
# 15f. 历史 S&P 500 成分变动
sp_hist_change = fmp('historical-sp500-constituent')
sp_hist_change.head(10)

---
## 16. 全球资产类别 (Forex / Crypto / Commodities)
实时汇率、加密货币报价和大宗商品数据。

In [ ]:
# === 16A. 外汇 (Forex) ===

# 16A-a. 外汇对列表
fx_list = fmp('forex-list')
print(f'可用外汇对数量: {len(fx_list)}')
fx_list.head(10)

In [ ]:
# 16A-b. 外汇实时报价
fx_quote = fmp('quote', symbol='EURUSD')
fx_quote

In [ ]:
# 16A-c. 外汇历史日线
fx_hist = fmp('historical-price-eod/full', symbol='EURUSD', **{'from': '2025-03-01', 'to': '2025-04-17'})
fx_hist.head(10)

In [ ]:
# === 16B. 加密货币 (Crypto) ===

# 16B-a. 加密货币列表
crypto_list = fmp('cryptocurrency-list')
print(f'可用加密货币数量: {len(crypto_list)}')
crypto_list.head(10)

In [ ]:
# 16B-b. BTC 实时报价
btc_quote = fmp('quote', symbol='BTCUSD')
btc_quote

In [ ]:
# 16B-c. BTC 历史日线
btc_hist = fmp('historical-price-eod/full', symbol='BTCUSD', **{'from': '2025-01-01', 'to': '2025-04-17'})
btc_hist.head(10)

In [ ]:
# === 16C. 大宗商品 (Commodities) ===

# 16C-a. 商品列表
comm_list = fmp('commodities-list')
print(f'可用商品数量: {len(comm_list)}')
comm_list.head(10)

In [ ]:
# 16C-b. 黄金实时报价
gold_quote = fmp('quote', symbol='GCUSD')
gold_quote

In [ ]:
# 16C-c. 原油历史走势
oil_hist = fmp('historical-price-eod/full', symbol='CLUSD', **{'from': '2025-01-01', 'to': '2025-04-17'})
oil_hist.head(10)

---
## 17. 批量获取能力 (Bulk Endpoints)
一次性下载全市场公司的财务快照、估值和收盘价。Premium 用户独享。

In [ ]:
# 17a. 全市场 EOD 收盘价 (Bulk EOD)
bulk_eod = fmp('eod-bulk', date='2025-04-16')
print(f'收录股票数量: {len(bulk_eod)}')
bulk_eod.head(10)

In [ ]:
# 17b. 全市场 Company Profile Bulk
bulk_profile = fmp('profile-bulk', part=0)
print(f'Profile records: {len(bulk_profile)}')
bulk_profile.head(5)

In [ ]:
# 17c. 全市场 DCF Bulk
bulk_dcf = fmp('dcf-bulk')
print(f'DCF records: {len(bulk_dcf)}')
bulk_dcf.head(10)

In [ ]:
# 17d. 全市场 Rating Bulk
bulk_rating = fmp('rating-bulk')
print(f'Rating records: {len(bulk_rating)}')
bulk_rating.head(10)

In [ ]:
# 17e. 全市场 Financial Scores Bulk
bulk_scores = fmp('scores-bulk')
print(f'Scores records: {len(bulk_scores)}')
bulk_scores.head(10)

In [ ]:
# 17f. 全市场 Key Metrics TTM Bulk
bulk_km = fmp('key-metrics-ttm-bulk')
print(f'Key Metrics TTM records: {len(bulk_km)}')
bulk_km.head(5)

In [ ]:
# 17g. 全市场 Income Statement Bulk (某季度)
bulk_income = fmp('income-statement-bulk', year=2024, period='Q4')
print(f'Income Statement records: {len(bulk_income)}')
bulk_income.head(5)

In [ ]:
# 17h. Price Target Summary Bulk
bulk_pt = fmp('price-target-summary-bulk')
print(f'Price Target records: {len(bulk_pt)}')
bulk_pt.head(10)

---
## 附加: 市场表现 & SEC 相关 (Bonus)
板块/行业表现、涨跌幅榜、SEC Filings 搜索。

In [ ]:
# 板块表现 (Sector Performance Snapshot)
sector_perf = fmp('sector-performance-snapshot', date='2025-04-16')
sector_perf

In [ ]:
# 行业 PE Snapshot
ind_pe = fmp('industry-pe-snapshot', date='2025-04-16')
ind_pe.head(10)

In [ ]:
# 今日涨幅榜 (Biggest Gainers)
gainers = fmp('biggest-gainers')
gainers.head(10)

In [ ]:
# 今日跌幅榜 (Biggest Losers)
losers = fmp('biggest-losers')
losers.head(10)

In [ ]:
# 最活跃股票 (Most Actives)
actives = fmp('most-actives')
actives.head(10)

In [ ]:
# SEC Filings 按公司搜索
sec_aapl = fmp('sec-filings-search/symbol', symbol='AAPL', **{'from': '2024-01-01', 'to': '2025-04-17'}, limit=10)
sec_aapl.head()

In [ ]:
# 交易所开市时间
market_hours = fmp('exchange-market-hours', exchange='NASDAQ')
market_hours

In [ ]:
# 公司Profile
profile = fmp('profile', symbol='AAPL')
profile.head()

In [ ]:
# 财报增长统计
growth = fmp('financial-growth', symbol='AAPL', period='annual', limit=5)
growth.head()

---
### 总结

| # | 类别 | 关键 endpoint 示例 |
|---|------|--------------------|
| 1 | 实时行情 | `quote`, `batch-quote`, `stock-price-change`, `aftermarket-*` |
| 2 | 财务报表 | `income-statement`, `balance-sheet-statement`, `cash-flow-statement`, `*-ttm`, `*-as-reported` |
| 3 | 估值指标 | `key-metrics`, `ratios`, `enterprise-values`, `owner-earnings`, `financial-scores` |
| 4 | DCF | `discounted-cash-flow`, `levered-discounted-cash-flow`, `custom-*` |
| 5 | 收入拆解 | `revenue-product-segmentation`, `revenue-geographic-segmentation` |
| 6 | 分析师 | `analyst-estimates`, `price-target-*`, `grades`, `ratings-snapshot` |
| 7 | 电话会议 | `earning-call-transcript`, `earning-call-transcript-dates` |
| 8 | 机构持仓 | `institutional-ownership/*` |
| 9 | 内幕交易 | `insider-trading/*` |
| 10 | 图表&技术 | `historical-price-eod/*`, `historical-chart/*`, `technical-indicators/*` |
| 11 | 另类数据 | `senate-*`, `house-*`, `esg-*`, `crowdfunding-*`, `commitment-of-traders-*` |
| 12 | 宏观经济 | `treasury-rates`, `economic-indicators`, `economic-calendar`, `market-risk-premium` |
| 13 | 日历 | `earnings-calendar`, `dividends-calendar`, `splits-calendar`, `ipos-calendar` |
| 14 | ETF/基金 | `etf/holdings`, `etf/info`, `etf/sector-weightings`, `etf/asset-exposure` |
| 15 | 指数 | `sp500-constituent`, `nasdaq-constituent`, `dowjones-constituent` |
| 16 | 全球资产 | `forex-list`, `cryptocurrency-list`, `commodities-list` + quote/chart |
| 17 | 批量获取 | `*-bulk` (eod, profile, dcf, rating, scores, key-metrics-ttm, income-statement...) |